### **NO-LEAK MODEL BENCHMARKING (PCA-only vs PCA+Extras)**
 - Evaluates multiple models on multiple augmented datasets
 - Temporal split: Day 1 = train, Day 2 = test
 - Threshold is selected ONLY on a validation split from train day
 - Preprocessing is fit ONLY on train (no leakage)
 - CatBoost uses native categorical handling (no one-hot)
 - Other models use OneHotEncoder fit on train only
 - Then: pick best (dataset, feature_set, model) by F2_test
 - Finally: hyperparameter tuning on the best config (CV on train day)


In [ ]:
import sys
!{sys.executable} -m pip install catboost

# Imports
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    fbeta_score,
    precision_score,
    recall_score
)

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import LocalOutlierFactor

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from imblearn.ensemble import BalancedRandomForestClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.2 MB/s eta 0:00:00


In [ ]:
colab = True
if colab is True:
  from google.colab import drive
  drive.mount("/content/drive")

  import pandas as pd
  from pathlib import Path

  # carpeta donde los has subido
  DATA_DIR = Path("/content/drive/MyDrive/UC3M/bluetab_df")


In [ ]:
# -----------------------------
# 0) Datasets
# -----------------------------
colab = True
if colab is True:
  from google.colab import drive
  drive.mount("/content/drive")

  import pandas as pd
  from pathlib import Path

  # carpeta donde los has subido
  DATA_DIR = Path("/content/drive/MyDrive/UC3M/bluetab_df")

  files = [
      "df_exp_50_2.csv",
      "df_exp_63_2.csv",
      "df_exp_random_2.csv",
      "df_exp_same_prop_2.csv",
  ]

  dfs = {}
  for f in files:
      p = DATA_DIR / f
      dfs[f] = pd.read_csv(p)
      print(f"Loaded {f}: shape={dfs[f].shape}")

  df_exp_50      = dfs["df_exp_50_2.csv"]
  df_exp_63      = dfs["df_exp_63_2.csv"]
  df_exp_random  = dfs["df_exp_random_2.csv"]
  df_exp_same    = dfs["df_exp_same_prop_2.csv"]
  DATASETS = [df_exp_50, df_exp_63, df_exp_random, df_exp_same]
else:
  DATASETS = [
      Path("csv_exports/df_exp_same_prop_2.csv"),
      Path("csv_exports/df_exp_random_2.csv"),
      Path("csv_exports/df_exp_63_2.csv"),
      Path("csv_exports/df_exp_50_2.csv"),
  ]

TARGET_COL = "Class"
TIME_COL = "timestamp"
DATE_COL = "date"

# IDs / leakage-ish columns you never want as predictors
ID_COLS = [
    "transaction_id", "customer_id", "device_id",
    "email", "phone", "name", "ip_address"
]

# Always drop these if present (metadata / split helpers)
DROP_ALWAYS = [TIME_COL, DATE_COL]

# PCA columns present in your exports (after dropping correlated ones earlier, you have only a subset)
PCA_COLS_ALL = [f"V{i}" for i in range(1, 29)]  # we will keep only those that exist in each df

# "Extras"  to add beyond PCA to improve performance
# (keep these stable across all datasets for fair comparison)
EXTRA_COLS = [
    "Amount",
    "amount_log",
    "customer_country",
    "merchant_country",
    "merchant",
    "city",
    "zip_code",
    "age",
    "credit_score",
    "days_in_bank",
    "days_in_bank_log",
    "secs_since_prev_tx",
    "tx_per_customer",
    "tx_rate",
    "hour_sin",
    "hour_cos",
    "is_weekend",
    "is_mobile",
    "is_foreign_tx",
    "is_night_tx",
    "is_high_amount",
    "geo_anomaly",
    "shared_ip",
    "device_change",
    "browser_change",
    "device_type",
    "os",
    "browser",
    "main_device",
    "main_browser",
    "time_of_day",
    "tenure_bucket",
    "night", "morning", "afternoon", "evening",
    "is_outlier_amount",
]

FEATURE_SETS = {
    "pca_only": "PCA-only",
    "pca_plus": "PCA + Extras",
}

TOPK_LIST = [5, 10, 15, 20, 30]


In [ ]:
# -----------------------------
# 1) Helpers: split + threshold
# -----------------------------
def temporal_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Day 1 -> train, Day 2 -> test (based on timestamp)."""
    out = df.copy()
    out[TIME_COL] = pd.to_datetime(out[TIME_COL], errors="coerce")
    out[DATE_COL] = out[TIME_COL].dt.date

    dates = sorted(out[DATE_COL].dropna().unique())
    if len(dates) < 2:
        raise ValueError("Need at least 2 different days for temporal split")

    train_date, test_date = dates[0], dates[1]
    train_df = out[out[DATE_COL] == train_date].copy()
    test_df = out[out[DATE_COL] == test_date].copy()
    return train_df, test_df


def best_f2_threshold(y_true, scores):
    """Pick threshold that maximizes F2 on validation data."""
    prec_arr, rec_arr, thr = precision_recall_curve(y_true, scores)
    best_thr, best_f2 = 0.5, 0.0

    # thr has length n-1 vs prec/rec length n
    for p, r, t in zip(prec_arr[:-1], rec_arr[:-1], thr):
        denom = (4 * p) + r
        f2 = (5 * (p * r) / denom) if denom > 0 else 0.0
        if f2 > best_f2:
            best_f2, best_thr = f2, t

    return float(best_thr), float(best_f2)


def get_scores(pipe, X):
    """Return probability-like scores for any estimator pipeline."""
    clf = pipe
    if hasattr(clf, "predict_proba"):
        return clf.predict_proba(X)[:, 1]
    if hasattr(clf, "decision_function"):
        return clf.decision_function(X)
    return clf.predict(X)

In [ ]:
# -----------------------------
# 2) Feature selection per df
# -----------------------------
def choose_feature_columns(
    df: pd.DataFrame,
    feature_set: str,
    extra_subset: list[str] | None = None,
) -> list[str]:
    """Return the columns that will be used as predictors for this df."""
    pca_cols, extras_all = get_available_feature_lists(df)

    if extra_subset is not None:
        extras = [c for c in extra_subset if c in extras_all]
    else:
        extras = extras_all

    if feature_set == "pca_only":
        chosen = pca_cols
    elif feature_set == "pca_plus":
        chosen = list(dict.fromkeys(pca_cols + extras))
    elif feature_set == "pca_topk":
        if extra_subset is None:
            raise ValueError("extra_subset must be provided for pca_topk")
        chosen = list(dict.fromkeys(pca_cols + extras))
    else:
        raise ValueError(f"Unknown feature_set={feature_set}")

    forbidden = set(ID_COLS + DROP_ALWAYS + [TARGET_COL])
    chosen = [c for c in chosen if c not in forbidden]

    if not chosen:
        raise ValueError("No usable predictor columns found for this df / feature_set")

    return chosen

In [ ]:
# -----------------------------
# 2b) Extra feature ranking (train-only)
# -----------------------------
def get_available_feature_lists(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    cols_present = set(df.columns)
    pca_cols = [c for c in PCA_COLS_ALL if c in cols_present]
    extras = [
        c for c in EXTRA_COLS
        if c in cols_present and c not in ID_COLS and c not in DROP_ALWAYS and c != TARGET_COL
    ]
    return pca_cols, extras


def fill_missing_for_ranking(X: pd.DataFrame):
    Xc = X.copy()
    cat_cols = Xc.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = Xc.select_dtypes(include=[np.number]).columns.tolist()

    for c in cat_cols:
        Xc[c] = Xc[c].astype(str).fillna("MISSING")
    for c in num_cols:
        med = Xc[c].median()
        Xc[c] = Xc[c].fillna(med)

    return Xc, cat_cols, num_cols

def rank_extras(train_df: pd.DataFrame) -> pd.DataFrame:
    """Return ranking of extra features using train-only signals (robust to categoricals)."""
    pca_cols, extras = get_available_feature_lists(train_df)
    if not extras:
        raise ValueError("No extra columns available for ranking")

    X = train_df[pca_cols + extras].copy()
    y = train_df[TARGET_COL].astype(int).copy()

    X_filled, cat_cols, num_cols = fill_missing_for_ranking(X)

    # --- Encode categoricals for TREE (LightGBM robust) ---
    X_tree = X_filled.copy()
    for c in cat_cols:
        X_tree[c] = X_tree[c].astype("category").cat.codes  # missing -> -1

    tree = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=-1,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    tree.fit(X_tree, y)

    tree_importance = pd.Series(tree.feature_importances_, index=X_tree.columns)
    tree_extras = tree_importance.reindex(extras).fillna(0)

    # --- MI (already encoded) ---
    X_mi = X_tree[extras].copy()
    discrete_mask = [(c in cat_cols) for c in X_mi.columns]
    mi_scores = pd.Series(
        mutual_info_classif(X_mi, y, discrete_features=discrete_mask, random_state=42),
        index=X_mi.columns,
    )

    # --- Correlation only for numeric extras ---
    corr_scores = {}
    for c in extras:
        if c in num_cols:
            corr_scores[c] = float(abs(pd.Series(X_filled[c]).corr(y)))
    corr_scores = pd.Series(corr_scores)

    ranking = pd.DataFrame({
        "tree_importance": tree_extras,
        "mi": mi_scores,
        "corr": corr_scores,
    })

    for col in ["tree_importance", "mi", "corr"]:
        ranking[f"{col}_rank"] = ranking[col].rank(ascending=False, method="dense")

    rank_cols = [c for c in ranking.columns if c.endswith("_rank")]
    ranking["mean_rank"] = ranking[rank_cols].mean(axis=1, skipna=True)

    return ranking.sort_values("mean_rank")




In [ ]:
# -----------------------------
# 3) Preprocessors (no leakage)
# -----------------------------
def build_preprocessor_for_non_catboost(X_train: pd.DataFrame):
    """OneHotEncoder fitted only on train; numeric impute; optional scale for LR/SVM."""
    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        # scaling is cheap and helps LR/SVM; trees ignore it but not harmful
        ("scaler", StandardScaler(with_mean=False)),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ])

    pre = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3
    )
    return pre


def prepare_catboost_data(X_train: pd.DataFrame, X_val: pd.DataFrame, X_test: pd.DataFrame):
    """For CatBoost: no OHE. Just basic imputing and return cat feature indices."""
    Xtr = X_train.copy()
    Xva = X_val.copy()
    Xte = X_test.copy()

    cat_cols = Xtr.select_dtypes(include=["object", "category"]).columns.tolist()
    cat_indices = [Xtr.columns.get_loc(c) for c in cat_cols]

    # Fill missing values
    for c in cat_cols:
        Xtr[c] = Xtr[c].astype(str).fillna("MISSING")
        Xva[c] = Xva[c].astype(str).fillna("MISSING")
        Xte[c] = Xte[c].astype(str).fillna("MISSING")

    num_cols = Xtr.select_dtypes(include=[np.number]).columns.tolist()
    for c in num_cols:
        med = Xtr[c].median()
        Xtr[c] = Xtr[c].fillna(med)
        Xva[c] = Xva[c].fillna(med)
        Xte[c] = Xte[c].fillna(med)

    return Xtr, Xva, Xte, cat_indices

In [ ]:
# -----------------------------
# 4) Models
# -----------------------------
def make_models():
    return {
        "lightgbm": lambda: LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=-1,
            num_leaves=63,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "xgboost": lambda: XGBClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            scale_pos_weight=10,
            random_state=42,
            n_jobs=-1,
        ),
        "catboost": lambda: CatBoostClassifier(
            iterations=400,
            learning_rate=0.05,
            depth=6,
            loss_function="Logloss",
            auto_class_weights="Balanced",
            random_seed=42,
            verbose=0,
            allow_writing_files=False,
        ),
        "random_forest": lambda: RandomForestClassifier(
            n_estimators=300,
            max_depth=20,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        "balanced_rf": lambda: BalancedRandomForestClassifier(
            n_estimators=300,
            max_depth=15,
            random_state=42,
            n_jobs=-1,
            sampling_strategy="auto",
        ),
        "svm_calibrated": lambda: CalibratedClassifierCV(
            estimator=LinearSVC(class_weight="balanced", max_iter=5000),
            method="sigmoid",
            cv=3,
        ),
        "logreg": lambda: LogisticRegression(
            penalty="l2",
            C=1.0,
            class_weight="balanced",
            solver="lbfgs",
            max_iter=2000,
            n_jobs=-1,
        ),
    }



def make_anomaly_models():
    return {
        "isolation_forest": IsolationForest(
            n_estimators=300,
            contamination="auto",
            random_state=42,
            n_jobs=-1,
        ),
        "lof": LocalOutlierFactor(
            n_neighbors=50,
            novelty=True,
            contamination="auto",
        ),
    }

In [ ]:
# -----------------------------
# 5) Evaluation: supervised
# -----------------------------
def eval_supervised(
    df: pd.DataFrame,
    dataset_name: str,
    feature_set: str,
    model_name: str,
    model,
    extra_subset: list[str] | None = None,
    feature_label: str | None = None,
    topk: int | None = None,
) -> dict:
    feature_label = feature_label or feature_set

    train_df, test_df = temporal_split(df)
    train_df, val_df = train_test_split(
        train_df, test_size=0.2, stratify=train_df[TARGET_COL], random_state=42
    )

    feats = choose_feature_columns(train_df, feature_set, extra_subset=extra_subset)

    X_train = train_df[feats].copy()
    y_train = train_df[TARGET_COL].astype(int).copy()
    X_val = val_df[feats].copy()
    y_val = val_df[TARGET_COL].astype(int).copy()
    X_test = test_df[feats].copy()
    y_test = test_df[TARGET_COL].astype(int).copy()

    # Adjust XGBoost scale_pos_weight using TRAIN only
    if model_name == "xgboost":
        pos = (y_train == 1).sum()
        neg = (y_train == 0).sum()
        spw = (neg / max(pos, 1))
        model.set_params(scale_pos_weight=float(spw))

    # --- CatBoost path: native categories (NO one-hot) ---
    if model_name == "catboost":
        Xtr, Xva, Xte, cat_idx = prepare_catboost_data(X_train, X_val, X_test)

        model.fit(Xtr, y_train, cat_features=cat_idx)
        val_scores = model.predict_proba(Xva)[:, 1]
        best_thr, best_f2_val = best_f2_threshold(y_val, val_scores)

        test_scores = model.predict_proba(Xte)[:, 1]
        y_pred = (test_scores >= best_thr).astype(int)

        return {
            "dataset": dataset_name,
            "feature_key": feature_set,
            "feature_label": feature_label,
            "topk": topk,
            "n_features_raw": len(feats),
            "model": model_name,
            "thr_val": best_thr,
            "f2_val": best_f2_val,
            "auc_pr_test": float(average_precision_score(y_test, test_scores)),
            "roc_auc_test": float(roc_auc_score(y_test, test_scores)),
            "f2_test": float(fbeta_score(y_test, y_pred, beta=2)),
            "precision_test": float(precision_score(y_test, y_pred, zero_division=0)),
            "recall_test": float(recall_score(y_test, y_pred, zero_division=0)),
        }

    # --- Non-CatBoost path: OneHotEncoder fit only on train ---
    pre = build_preprocessor_for_non_catboost(X_train)
    pipe = Pipeline([("prep", pre), ("clf", model)])
    pipe.fit(X_train, y_train)

    val_scores = get_scores(pipe, X_val)
    best_thr, best_f2_val = best_f2_threshold(y_val, val_scores)

    test_scores = get_scores(pipe, X_test)
    y_pred = (test_scores >= best_thr).astype(int)

    return {
        "dataset": dataset_name,
        "feature_key": feature_set,
        "feature_label": feature_label,
        "topk": topk,
        "n_features_raw": len(feats),
        "model": model_name,
        "thr_val": best_thr,
        "f2_val": best_f2_val,
        "auc_pr_test": float(average_precision_score(y_test, test_scores)),
        "roc_auc_test": float(roc_auc_score(y_test, test_scores)),
        "f2_test": float(fbeta_score(y_test, y_pred, beta=2)),
        "precision_test": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall_test": float(recall_score(y_test, y_pred, zero_division=0)),
    }

In [ ]:
# -----------------------------
# 6) Evaluation: anomaly (unsupervised)
# -----------------------------
def eval_anomaly(df: pd.DataFrame, dataset_name: str, feature_set: str, model_name: str, model) -> dict:
    train_df, test_df = temporal_split(df)
    train_df, val_df = train_test_split(
        train_df, test_size=0.2, stratify=train_df[TARGET_COL], random_state=42
    )

    feats = choose_feature_columns(train_df, feature_set)

    X_train = train_df[feats].copy()
    y_train = train_df[TARGET_COL].astype(int).copy()
    X_val = val_df[feats].copy()
    y_val = val_df[TARGET_COL].astype(int).copy()
    X_test = test_df[feats].copy()
    y_test = test_df[TARGET_COL].astype(int).copy()

    # For anomaly models, we must produce a "score": higher => more anomalous
    pre = build_preprocessor_for_non_catboost(X_train)
    pipe = Pipeline([("prep", pre), ("clf", model)])

    # Fit without labels
    pipe.fit(X_train)

    def anomaly_scores(clf, X):
        est = clf.named_steps["clf"]
        Xt = clf.named_steps["prep"].transform(X)
        # IsolationForest has score_samples; LOF has decision_function (novelty=True)
        if hasattr(est, "decision_function"):
            return -est.decision_function(Xt)
        if hasattr(est, "score_samples"):
            return -est.score_samples(Xt)
        # fallback
        return -est.predict(Xt)

    val_scores = anomaly_scores(pipe, X_val)
    best_thr, best_f2_val = best_f2_threshold(y_val, val_scores)

    test_scores = anomaly_scores(pipe, X_test)
    y_pred = (test_scores >= best_thr).astype(int)

    return {
        "dataset": dataset_name,
        "feature_set": feature_set,
        "n_features_raw": len(feats),
        "model": model_name,
        "thr_val": best_thr,
        "f2_val": best_f2_val,
        "auc_pr_test": float(average_precision_score(y_test, test_scores)),
        "roc_auc_test": float(roc_auc_score(y_test, test_scores)),
        "f2_test": float(fbeta_score(y_test, y_pred, beta=2)),
        "precision_test": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall_test": float(recall_score(y_test, y_pred, zero_division=0)),
    }


In [ ]:
# -----------------------------
# 7) Run benchmark
# -----------------------------
results = []
models = make_models()
anomaly_models = make_anomaly_models()

for path in DATASETS:
    df = pd.read_csv(path)
    dataset_name = path.name
    print(f"\n==== DATASET: {dataset_name} ====")

    train_day_df, _ = temporal_split(df)
    extra_ranking = rank_extras(train_day_df)
    print("\nTop 15 extras by mean rank:")
    display(extra_ranking.head(15))

    topk_map = {}
    for k in TOPK_LIST:
        k_adj = min(k, len(extra_ranking))
        topk_map[k] = extra_ranking.index.tolist()[:k_adj]

    for feature_set in FEATURE_SETS.keys():
        for m_name, m in models.items():
            print(f"\n=== SUP | {dataset_name} | {FEATURE_SETS[feature_set]} | {m_name} ===")
            try:
                model = m()
                res = eval_supervised(df, dataset_name, feature_set, m_name, model)
                results.append(res)
                print(res)
            except Exception as e:
                print(f"Error: {e}")

        for m_name, m in anomaly_models.items():
            print(f"\n=== ANOM | {dataset_name} | {FEATURE_SETS[feature_set]} | {m_name} ===")
            try:
                res = eval_anomaly(df, dataset_name, feature_set, m_name, m)
                results.append(res)
                print(res)
            except Exception as e:
                print(f"Error: {e}")

    for k, extra_subset in topk_map.items():
        feature_label = f"PCA + top{k} extras"
        for m_name, m in models.items():
            print(f"\n=== SUP | {dataset_name} | {feature_label} | {m_name} ===")
            try:
                res = eval_supervised(
                    df,
                    dataset_name,
                    "pca_topk",
                    m_name,
                    m,
                    extra_subset=extra_subset,
                    feature_label=feature_label,
                    topk=k,
                )
                results.append(res)
                print(res)
            except Exception as e:
                print(f"Error: {e}")

results_df = pd.DataFrame(results)
display(results_df.sort_values(["f2_test", "f2_val"], ascending=False))



==== DATASET: df_exp_same_prop_2.csv ====
[LightGBM] [Info] Number of positive: 2821, number of negative: 356630
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.076226 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7465
[LightGBM] [Info] Number of data points in the train set: 359451, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Users\M6500QC\OneDrive\Escritorio\1º Cuatrimestre\DATA SCIENCE PROJECT\bluetab-uc3m-project-repo\.venv\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\M6500QC\OneDrive\Escritorio\1º Cuatrimestre\DATA SCIENCE PROJECT\bluetab-uc3m-project-repo\.venv\lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



Top 15 extras by mean rank:


,tree_importance,mi,corr,tree_importance_rank,mi_rank,corr_rank,mean_rank
amount_log,1160,0.013350,0.021173,1.0,12.0,6.0,6.333333
merchant,68,0.035071,NaN,7.0,6.0,NaN,6.500000
zip_code,92,0.032621,NaN,6.0,7.0,NaN,6.500000
hour_sin,712,0.002251,0.034163,2.0,16.0,2.0,6.666667
evening,59,0.029268,0.024070,9.0,8.0,4.0,7.000000
morning,63,0.014152,0.013576,8.0,10.0,7.0,8.333333
Amount,269,0.010840,0.006292,4.0,13.0,9.0,8.666667
hour_cos,597,0.001650,0.002918,3.0,18.0,11.0,10.666667
secs_since_prev_tx,46,0.001427,0.030302,13.0,19.0,3.0,11.666667
is_night_tx,24,0.002245,0.038738,18.0,17.0,1.0,12.000000



=== SUP | df_exp_same_prop_2.csv | PCA-only | lightgbm ===
[LightGBM] [Info] Number of positive: 2257, number of negative: 285303
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020075 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3825
[LightGBM] [Info] Number of data points in the train set: 287560, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Users\M6500QC\OneDrive\Escritorio\1º Cuatrimestre\DATA SCIENCE PROJECT\bluetab-uc3m-project-repo\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\M6500QC\OneDrive\Escritorio\1º Cuatrimestre\DATA SCIENCE PROJECT\bluetab-uc3m-project-repo\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'dataset': 'df_exp_same_prop_2.csv', 'feature_key': 'pca_only', 'feature_label': 'pca_only', 'topk': None, 'n_features_raw': 15, 'model': 'lightgbm', 'thr_val': 0.08021543096162986, 'f2_val': 0.9971611071682044, 'auc_pr_test': 0.7720962434505747, 'roc_auc_test': 0.9548989737405598, 'f2_test': 0.7550295857988165, 'precision_test': 0.8045397225725095, 'recall_test': 0.7435897435897436}

=== SUP | df_exp_same_prop_2.csv | PCA-only | xgboost ===
{'dataset': 'df_exp_same_prop_2.csv', 'feature_key': 'pca_only', 'feature_label': 'pca_only', 'topk': None, 'n_features_raw': 15, 'model': 'xgboost', 'thr_val': 0.39860057830810547, 'f2_val': 0.9982269503546101, 'auc_pr_test': 0.7828986710061178, 'roc_auc_test': 0.9626824032977614, 'f2_test': 0.7604017216642754, 'precision_test': 0.848, 'recall_test': 0.7412587412587412}

=== SUP | df_exp_same_prop_2.csv | PCA-only | catboost ===
{'dataset': 'df_exp_same_prop_2.csv', 'feature_key': 'pca_only', 'feature_label': 'pca_only', 'topk': None, 'n_features

In [ ]:
# ============================================================
# 8) Hyperparameter tuning for the BEST (dataset, feature_set, model)
# - CV and threshold selection happen ONLY inside train day (no test leakage)
# - After tuning, you evaluate ONCE on the untouched test day
# ============================================================

best_row = results_df.sort_values("f2_test", ascending=False).iloc[0]
print("\nBEST CONFIG FOUND:")
display(best_row)

best_dataset = best_row["dataset"]
best_feature_key = best_row["feature_key"]         # pca_only / pca_plus / pca_topk
best_feature_label = best_row["feature_label"]     # texto bonito
best_topk = best_row.get("topk", None)
best_model_name = best_row["model"]

# Reload that dataset
best_df = pd.read_csv(Path("csv_exports") / best_dataset)

# Rebuild train/test (Day1/Day2)
train_df_full, test_df = temporal_split(best_df)

# Validation split inside train day
train_df, val_df = train_test_split(
    train_df_full, test_size=0.2, stratify=train_df_full[TARGET_COL], random_state=42
)

# --- Rebuild feature list WITHOUT leakage ---
extra_subset = None
if best_feature_key == "pca_topk":
    # Rank extras using ONLY train day (and ideally only train_df_full; no test leakage)
    extra_ranking = rank_extras(train_df_full)
    k = int(best_topk) if pd.notna(best_topk) else 0
    k = min(k, len(extra_ranking))
    extra_subset = extra_ranking.index.tolist()[:k]

feats = choose_feature_columns(
    train_df,                      # columns available
    best_feature_key,              # key interno
    extra_subset=extra_subset      # solo si topk
)

X_train = train_df[feats].copy()
y_train = train_df[TARGET_COL].astype(int).copy()
X_val = val_df[feats].copy()
y_val = val_df[TARGET_COL].astype(int).copy()
X_test = test_df[feats].copy()
y_test = test_df[TARGET_COL].astype(int).copy()


# Tune only a couple of models (typical best ones); here example for RandomForest or LightGBM
if best_model_name == "random_forest":
    base = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced")

    # Preprocess with OHE for non-catboost
    pre = build_preprocessor_for_non_catboost(X_train)
    pipe = Pipeline([("prep", pre), ("clf", base)])

    # Randomized search space
    param_dist = {
        "clf__n_estimators": [200, 300, 500],
        "clf__max_depth": [10, 15, 20, None],
        "clf__min_samples_split": [2, 5, 10],
        "clf__min_samples_leaf": [1, 2, 4],
        "clf__max_features": ["sqrt", "log2"],
    }

    f2_scorer = lambda yt, yp: fbeta_score(yt, yp, beta=2)
    # scoring needs predicted labels: we will tune on default threshold 0.5 in CV
    # (final threshold will still be chosen on val later)
    search = RandomizedSearchCV(
        pipe,
        param_distributions=param_dist,
        n_iter=12,
        cv=StratifiedKFold(3, shuffle=True, random_state=42),
        scoring="f1",  # stable surrogate; final selection uses F2 via val threshold
        n_jobs=-1,
        random_state=42,
        verbose=1,
    )

    search.fit(X_train, y_train)
    best_pipe = search.best_estimator_
    print("\nBest RF params:", search.best_params_)

    # choose threshold on val (no leak)
    val_scores = get_scores(best_pipe, X_val)
    best_thr, best_f2_val = best_f2_threshold(y_val, val_scores)

    # evaluate on test once
    test_scores = get_scores(best_pipe, X_test)
    y_pred = (test_scores >= best_thr).astype(int)

    tuned_summary = {
        "best_model": "random_forest_tuned",
        "best_params": search.best_params_,
        "thr_val": best_thr,
        "f2_val": best_f2_val,
        "auc_pr_test": float(average_precision_score(y_test, test_scores)),
        "roc_auc_test": float(roc_auc_score(y_test, test_scores)),
        "f2_test": float(fbeta_score(y_test, y_pred, beta=2)),
        "precision_test": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall_test": float(recall_score(y_test, y_pred, zero_division=0)),
    }
    print("\nTUNED RESULTS:")
    print(tuned_summary)

else:
    print("\nTuning template included for RandomForest only in this snippet.")
    print("If your best model is LightGBM/XGBoost/CatBoost, tell me which one won and I adapt the tuning block.")
